In [ ]:
# Appliance Electricity Bill Estimator
# Finds which appliance uses the most electricity and estimates the monthly bill.
#
# Formula:  units (kWh) = watts x hours per day x quantity x days / 1000

import csv
import matplotlib.pyplot as plt

FILE_NAME = r"C:\Users\razee\Downloads\electricity-bill\electricity-bill\appliances.csv"

# Each slab is (up to how many units, energy rate per unit in rupees)
# ---------------------------------------------------------------------
SLABS = [(100, 3.96), (300, 10.80), (500, 15.03), (100000, 17.53)]
WHEELING_RATE = 1.60        # rupees per unit, charged on ALL units
FIXED_CHARGE = 130          # rupees per month (single phase; three phase is 435)
FAC_RATE = 0.0              # fuel adjustment charge per unit; it changes every
                            # month, so copy it from your bill (0 = ignore it)
DUTY_PERCENT = 16           # electricity duty on the bill (percent)
DAYS_IN_MONTH = 30


# ---------- PART 1: FILE HANDLING ----------

def load_appliances():
    """Read appliances.csv and return a list of dictionaries."""
    appliances = []
    try:
        file = open(FILE_NAME, "r")
        reader = csv.reader(file)
        next(reader)                       # skip the heading line
        for row in reader:
            try:
                appliance = {
                    "name": row[0],
                    "watts": float(row[1]),
                    "hours": float(row[2]),
                    "quantity": int(row[3]),
                }
                appliances.append(appliance)
            except (ValueError, IndexError):
                print("Skipping a bad row:", row)
        file.close()
    except FileNotFoundError:
        print("appliances.csv not found. Starting with an empty list.")
    return appliances


def save_appliances(appliances):
    file = open(FILE_NAME, "w", newline="")
    writer = csv.writer(file)
    writer.writerow(["Name", "Watts", "Hours_per_day", "Quantity"])
    for a in appliances:
        writer.writerow([a["name"], a["watts"], a["hours"], a["quantity"]])
    file.close()


# ---------- PART 2: CALCULATIONS ----------

def monthly_units(appliance):
    """Units (kWh) one appliance uses in a month."""
    return appliance["watts"] * appliance["hours"] * appliance["quantity"] * DAYS_IN_MONTH / 1000


def total_units(appliances):
    total = 0
    for a in appliances:
        total = total + monthly_units(a)
    return total


def energy_charge(units):
    """Slab-wise charge: the first units are cheaper, the later units cost more."""
    charge = 0
    previous_limit = 0
    for limit, rate in SLABS:
        if units > limit:
            charge = charge + (limit - previous_limit) * rate    # slab is full
            previous_limit = limit
        else:
            charge = charge + (units - previous_limit) * rate    # last slab used
            break
    return charge


def calculate_bill(units):
    """Return the bill as a dictionary, so each part can be shown by name."""
    energy = energy_charge(units)
    wheeling = units * WHEELING_RATE
    fac = units * FAC_RATE
    subtotal = energy + wheeling + fac + FIXED_CHARGE
    duty = subtotal * DUTY_PERCENT / 100
    return {"energy": energy, "wheeling": wheeling, "fac": fac,
            "fixed": FIXED_CHARGE, "duty": duty, "total": subtotal + duty}


def find_appliance(appliances, name):
    """Return the position of an appliance by name, or -1 if not found."""
    position = 0
    for a in appliances:
        if a["name"].lower() == name.lower():
            return position
        position = position + 1
    return -1


# ---------- PART 3: MENU OPTIONS ----------

def view_appliances(appliances):
    if len(appliances) == 0:
        print("No appliances yet.")
        return
    print("No.  Name                      Watts  Hours/day  Qty")
    number = 1
    for a in appliances:
        print(number, "  ", a["name"].ljust(25), a["watts"], "  ", a["hours"], "  ", a["quantity"])
        number = number + 1


def add_appliance(appliances):
    name = input("Appliance name: ").title()
    try:
        watts = float(input("Power in watts (see label on appliance): "))
        hours = float(input("Hours used per day: "))
        quantity = int(input("How many such appliances: "))
    except ValueError:
        print("Please enter numbers only.")
        return
    if watts <= 0 or hours < 0 or hours > 24 or quantity <= 0:
        print("Invalid values. Hours must be between 0 and 24.")
        return
    appliances.append({"name": name, "watts": watts, "hours": hours, "quantity": quantity})
    save_appliances(appliances)
    print("Added!")


def remove_appliance(appliances):
    view_appliances(appliances)
    name = input("Name of appliance to remove: ")
    position = find_appliance(appliances, name)
    if position == -1:
        print("Appliance not found.")
    else:
        appliances.pop(position)
        save_appliances(appliances)
        print("Removed.")


def change_hours(appliances):
    name = input("Name of appliance: ")
    position = find_appliance(appliances, name)
    if position == -1:
        print("Appliance not found.")
        return
    try:
        hours = float(input("New hours per day: "))
    except ValueError:
        print("Please enter a number.")
        return
    if hours < 0 or hours > 24:
        print("Hours must be between 0 and 24.")
        return
    appliances[position]["hours"] = hours
    save_appliances(appliances)
    print("Updated.")


def show_breakdown(appliances):
    """Show which appliance uses how much electricity, biggest first."""
    if len(appliances) == 0:
        print("No appliances yet.")
        return
    units = total_units(appliances)
    if units == 0:
        print("All appliances have 0 hours, so there is nothing to compare.")
        return
    total = calculate_bill(units)["total"]

    rows = []
    for a in appliances:
        rows.append((monthly_units(a), a["name"]))
    rows.sort(reverse=True)               # biggest consumer first

    print("Appliance                 Units(kWh)  Share%   Approx cost(Rs)")
    for kwh, name in rows:
        share = kwh / units * 100
        cost = total * kwh / units         # share of the total bill
        print(name.ljust(25), str(round(kwh, 1)).ljust(11), str(round(share, 1)).ljust(8), round(cost))
    print("-" * 60)
    print("Total units:", round(units, 1), "kWh")

    top_units, top_name = rows[0]
    print()
    print("Biggest consumer:", top_name, "-", round(top_units / units * 100), "% of your electricity")
    if "air" in top_name.lower() or "ac" in top_name.lower().split():
        print("Tip: set the AC to 24-26 C and use a timer. Each degree higher saves electricity.")
    else:
        print("Tip: reduce the hours of", top_name, "or buy a higher star-rated one.")


def show_bill(appliances):
    units = total_units(appliances)
    bill = calculate_bill(units)
    print("---------- ESTIMATED MONTHLY BILL (MSEDCL) ----------")
    print("Units used       :", round(units, 1), "kWh")
    print("Energy charge    : Rs", round(bill["energy"], 2))
    print("Wheeling charge  : Rs", round(bill["wheeling"], 2))
    if FAC_RATE > 0:
        print("Fuel adjustment  : Rs", round(bill["fac"], 2))
    print("Fixed charge     : Rs", bill["fixed"])
    print("Electricity duty : Rs", round(bill["duty"], 2))
    print("TOTAL BILL       : Rs", round(bill["total"], 2))
    print("-----------------------------------------------------")
    print("Estimate using FY 2026-27 slab rates set at the top of the code.")


def compare_with_real_bill(appliances):
    """Check how close our estimate is to the real meter reading."""
    try:
        actual = float(input("Units shown on your real bill: "))
    except ValueError:
        print("Please enter a number.")
        return
    estimated = total_units(appliances)
    difference = estimated - actual
    percent = 0
    if actual > 0:
        percent = difference / actual * 100
    print("Estimated:", round(estimated, 1), " Actual:", actual)
    if abs(percent) <= 10:
        print("Very close! Difference is", round(abs(percent), 1), "%")
    elif difference > 0:
        print("We estimated", round(percent), "% MORE. Check if the hours are too high.")
    else:
        print("We estimated", round(abs(percent)), "% LESS. You may have missed some appliances.")


def what_if(appliances):
    """How much money will I save if I use an appliance fewer hours?"""
    name = input("Which appliance? ")
    position = find_appliance(appliances, name)
    if position == -1:
        print("Appliance not found.")
        return
    try:
        new_hours = float(input("Try new hours per day: "))
    except ValueError:
        print("Please enter a number.")
        return
    a = appliances[position]
    old_units = total_units(appliances)
    old_part = monthly_units(a)
    new_part = a["watts"] * new_hours * a["quantity"] * DAYS_IN_MONTH / 1000
    new_units = old_units - old_part + new_part
    old_bill = calculate_bill(old_units)["total"]
    new_bill = calculate_bill(new_units)["total"]
    print("Old bill: Rs", round(old_bill), "  New bill: Rs", round(new_bill))
    saving = old_bill - new_bill
    if saving > 0:
        print("You save Rs", round(saving), "per month, Rs", round(saving * 12), "per year!")
    else:
        print("Your bill increases by Rs", round(-saving))


# ---------- PART 4: CHART ----------

def show_chart(appliances):
    if len(appliances) == 0:
        print("No appliances yet.")
        return
    names = []
    units = []
    for a in appliances:
        names.append(a["name"])
        units.append(monthly_units(a))
    plt.figure(figsize=(9, 5))
    plt.barh(names, units)
    plt.xlabel("Units used per month (kWh)")
    plt.title("Electricity used by each appliance")
    plt.tight_layout()
    plt.savefig("appliance_usage.png")
    print("Chart saved as appliance_usage.png. Close the window to continue.")
    plt.show()


# ---------- PART 5: MAIN PROGRAM ----------

def main():
    appliances = load_appliances()
    print("Loaded", len(appliances), "appliances.")

    while True:
        print()
        print("===== ELECTRICITY BILL ESTIMATOR =====")
        print("1. View appliances")
        print("2. Add appliance")
        print("3. Remove appliance")
        print("4. Change hours of an appliance")
        print("5. Which appliance uses the most?")
        print("6. Estimated monthly bill")
        print("7. What if I use an appliance less?")
        print("8. Compare with my real bill")
        print("9. Show chart")
        print("0. Exit")
        choice = input("Choose: ")

        if choice == "1":
            view_appliances(appliances)
        elif choice == "2":
            add_appliance(appliances)
        elif choice == "3":
            remove_appliance(appliances)
        elif choice == "4":
            change_hours(appliances)
        elif choice == "5":
            show_breakdown(appliances)
        elif choice == "6":
            show_bill(appliances)
        elif choice == "7":
            what_if(appliances)
        elif choice == "8":
            compare_with_real_bill(appliances)
        elif choice == "9":
            show_chart(appliances)
        elif choice == "0":
            print("Goodbye!")
            break
        else:
            print("Invalid choice. Try again.")


if __name__ == "__main__":
    main()
